## Scope Connect and Setup

In [ ]:
import chipwhisperer as cw
scope = cw.scope()
scope.default_setup()

## Target Connect

In [ ]:
target = cw.target(scope)

## Object Info

In [ ]:
print(scope) # also works with project, CPA, etc.

## Build Firmware

In [ ]:
%%bash
cd ../firmware/mcu/simpleserial-aes
make -j PLATFORM=CWHUSKY CRYPTO_TARGET=TINYAES128C

## Program Target

In [ ]:
cw.program_target(scope, cw.programmers.SAM4SProgrammer, "../firmware/mcu/simpleserial-aes/simpleserial-aes-CWHUSKY.hex")

## Capture Loop

In [ ]:
from tqdm.notebook import tqdm

project = cw.Project()

N = 100
for key, text, n in tqdm(cw.ktp.BasicIt(N), desc='Capturing traces'):
    container = cw.capture_trace(scope, target, text, key)
    if container is not None:
        project.append(container)

## Save Project to Zip

In [ ]:
project.save("/path/to/file.zip")

## Project Data Access

In [ ]:
project.traces[:] # traces
project.plaintexts[:] # plaintexts
project.ciphertexts[:] # ciphertexets
project.keys[:] # keys
project[:] # project as trace container

## Capture Loop with Real Time Plotting

In [ ]:
plt = cw.StreamPlot()
plt.plot()

In [ ]:
from tqdm.notebook import tqdm

project = cw.Project()

N = 100
for key, text, n in tqdm(cw.ktp.BasicIt(N), desc='Capturing traces'):
    container = cw.capture_trace(scope, target, text, key)
    if container is not None:
        project.append(container)
        plt.update(container.trace)

## Basic Attack Using Analyzer

In [ ]:
from chipwhisperer.analyzer import CPA, get_table_cb, leakage_models

cb = get_table_cb()
cpa = CPA(project, leakage_models.sbox_output, 16)
cpa.run(10, callback=cb)
print(cpa.key_guess())

## Plotting Analyzer Info

In [ ]:
cpa.corr_v_time_plot() # or pge_v_traces_plot() or corr_v_traces_plot()